# BCU AI Hackathon 2026 — Team Apexmind

Retrieval-Augmented Generation pipeline for the 100-question multiple-choice set.

- **Model:** `qwen2.5:7b` (7B) primary + `qwen2.5:3b` (3B) tie-breaker — both **within the 8B limit**
- **Backend:** auto-detected — **Ollama** if a local server is reachable, otherwise **Hugging Face transformers** (e.g. on a Colab GPU)
- **Evidence:** live Wikipedia (MediaWiki API) + DuckDuckGo fallback, ranked with BM25
- **Output:** `Apexmind_submission.csv`

Run the cells top to bottom. On Colab, set `REPO_URL` in the *Get code & data* cell.

## 1. Install dependencies

In [ ]:
# Core pipeline deps
!pip -q install pandas requests rank-bm25 ddgs

# Needed ONLY for the transformers backend (the default on Colab, where Ollama
# is absent). Safe to keep installed even if you use the Ollama backend.
!pip -q install "transformers>=4.44" accelerate torch

## 2. Get code & data

If you are on Colab, set `REPO_URL` to your GitHub repo so the notebook can pull
`starter_code/run.py`, `starter_code/numeric_pass.py` and `questions_100.csv`.
If you are running inside the repo already, this cell just locates the files.

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/<your-username>/<your-repo>.git"  # <-- EDIT on Colab

def find_repo_root():
    p = Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / "questions_100.csv").exists():
            return cand
    return None

root = find_repo_root()
if root is None:
    # Not inside the repo (fresh Colab) -> clone it.
    !git clone $REPO_URL _repo
    root = Path.cwd() / "_repo"

os.chdir(root)
sys.path.insert(0, str(root / "starter_code"))
print("Repo root:", root)
print("questions file present:", (root / "questions_100.csv").exists())

## 3. (Optional) Use the Ollama backend on Colab

By default, with no Ollama server present, the pipeline uses **transformers** and
downloads the Qwen models from Hugging Face — nothing extra needed.

Run the cell below **only** if you specifically want the Ollama backend on Colab.

In [ ]:
# OPTIONAL — install + start Ollama and pull the models, then the pipeline
# will auto-detect and use it.
# !curl -fsSL https://ollama.com/install.sh | sh
# import subprocess, time
# subprocess.Popen(["ollama", "serve"]); time.sleep(5)
# !ollama pull qwen2.5:7b && ollama pull qwen2.5:3b

## 4. Run the pipeline

`run_pipeline` retrieves evidence, ranks it, and answers all 100 questions
(7B model + 3B tie-breaker). `numeric_pass` then fixes number/unit
disambiguation. This writes `Apexmind_submission.csv`.

> First call loads the model — on a Colab GPU expect a few minutes of download/warm-up.

In [ ]:
import run, numeric_pass

run.run_pipeline(run.DEFAULT_QUESTIONS_FILE, run.DEFAULT_OUTPUT_FILE)
numeric_pass.main()

## 5. Inspect & download the submission

In [ ]:
import pandas as pd
df = pd.read_csv(run.DEFAULT_OUTPUT_FILE)
print(len(df), "rows |", "all valid:", df["answer"].isin(list("ABCDE") + ["Unknown"]).all())
print(df["answer"].value_counts().to_dict())
df.head(10)

In [ ]:
# On Colab, uncomment to download the file:
# from google.colab import files
# files.download(str(run.DEFAULT_OUTPUT_FILE))